# How to assign context meaning to vector embeddings?

## 3.3.1 A simple self-attention mechanism without trainable weights

The key idea behind this is: how do we assign context meaning to a vector embedding? The result is the context vector.

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

/Users/yuanliheng/Desktop/Tech-Projects/projects/llm/build_your_llm_from_scratch/.venv/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
# An example of computing unnormalized attention scores using the 2nd token as query
# The attention score for a specific token regarding to the query is the dot product between the query and the token
query = inputs[1]
attention_scores_2 = torch.empty(inputs.shape[0])
for i, token in enumerate(inputs):
    attention_scores_2[i] = torch.dot(token, query)

print(attention_scores_2)


tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [3]:
# Normalize the unnormalized attention scores so that they sum up to 1
# note: dim=0 means apply softmax along the rows
attn_weights_2 = torch.softmax(attention_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [4]:
# Compute the context vector by multiplying the input token with the attention weights and sum the resulting vectors
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)

for i, token in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * token

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## 3.3.2 Computing attention weights for all input tokens

In [5]:
# compute unnormalized attention scores
attention_scores = torch.empty(6,6)

for i, query in enumerate(inputs):
    for j, token in enumerate(inputs):
        attention_scores[j, i] = torch.dot(query, token)

print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [6]:
# The same could be achieved via matrix multiplication
attention_scores = inputs @ inputs.T
print(attention_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [7]:
# normalize the attention scores
attn_weights = torch.softmax(attention_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [8]:
# compute context vectors
context_vectors = attn_weights @ inputs
print(context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


# 3.4 Implementing self-attention with trainable weights

In [ ]:
# We first show an example of computing context vectors for 1 query
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [ ]:
# Step 1: create Weight matrices for query, key and value
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [ ]:
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

In [ ]:
# Here's an analogy to understand keys, values and queries
# keys: what does the token contain?
# values: what information could the token provide?
# queries: what is the token looking for

keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [17]:
# Step 2: we compute the unnormalized attention scores by computing the dot product of the query vector and each key vector
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [18]:
# Step 3: we compute the attention weights (normalized attention scores that sum up to 1) using the softmax function.
# The difference to earlier is:
# we now scale the attention scores by dividing them by the square root of the embedding dimension. 

d_k = keys.shape[1]
attn_weights = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [19]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3069, 0.8188])
